# Cleanup Skript nach Woche 4
- Zeilenreduktion per `thresh`: entfernen von Zeilen mit sehr vielen Missing Values, wir nehmen mind. 90 ausgefüllte Spalten, was unserem Ausgangsdatensatz nach Woche 4 ca. 61% entspricht
- Spaltenbereinigung: Droppen vieler nicht benötigter Features
- Text-Normalisierung:
    - Vereinheitlichung von Textspalten (lowercase)
    - Bereinigung von Multi-Select-Spalten (Split per Semikolon, Whitespace-Cleanup, Duplikate entfernen, sortieren und wieder zusammenfügen)
- Numerische Konvertierungen: `YearsCode` und `WorkExp` in numerische Werte überführt um sie später analysier- und modellierbar zu machen
- Plausabilitätschecks: Filterlogik, um unplausible Kombinationen (bswp. Erfahrung > Alter) zu entfernen
- Outlier-Handling: entfernen extremer Ausreißer und "Quatschwerte" über dem 98% Quantil für ausgewählte numerische Spalten
- abschließend mit gecleantem Datensatz wieder mit `thresh` eine Zeilenreduktion durchführen, um final bereinigten Datensatz zu speichern
    - hier wurden von möglichen 38 Spalten 32 als `thresh`-Wert gewählt, was ca. 84% ausgefüllte Spalten entspricht
- der abschließende Shape des Datensatzes umfasst 18603 Zeilen und 38 Spalten

In [1]:
import re

import numpy as np
import pandas as pd
import unicodedata

In [2]:
df = pd.read_csv("output_woche4.csv")
df.info()
df.ConvertedCompTotal.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47803 entries, 0 to 47802
Columns: 147 entries, Unnamed: 0 to RemoteCategoryNum
dtypes: float64(41), int64(2), object(104)
memory usage: 53.6+ MB


count    2.234200e+04
mean     2.486597e+70
std      3.716777e+72
min      0.000000e+00
25%      4.204080e+04
50%      7.878925e+04
75%      1.250000e+05
max      5.555556e+74
Name: ConvertedCompTotal, dtype: float64

In [3]:
thresh_rows = 90  # mindestens 90 ausgefüllte Spalten pro Zeile
df_rows = df.dropna(axis=0, thresh=thresh_rows)

print("vorher:", df.shape)
print("nachher:", df_rows.shape)
df_rows.info()
df_rows.to_csv("survey_results_public_reduced.csv", index=False)
print("Saved:", df_rows.shape, "->", "survey_results_public_reduced.csv")

vorher: (47803, 147)
nachher: (22154, 147)
<class 'pandas.core.frame.DataFrame'>
Index: 22154 entries, 0 to 47801
Columns: 147 entries, Unnamed: 0 to RemoteCategoryNum
dtypes: float64(41), int64(2), object(104)
memory usage: 25.0+ MB
Saved: (22154, 147) -> survey_results_public_reduced.csv


In [4]:
IN_PATH = "survey_results_public_reduced.csv"
OUT_PATH = "survey_results_cleaned.csv"

df = pd.read_csv(IN_PATH)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22154 entries, 0 to 22153
Columns: 147 entries, Unnamed: 0 to RemoteCategoryNum
dtypes: float64(41), int64(2), object(104)
memory usage: 24.8+ MB


In [5]:
drop_cols = [
    'EmploymentAddl', 'LearnCodeChoose', 'LearnCode', 'AILearnHow', 'PurchaseInfluence',
    'ToolCountWork', 'ToolCountPersonal',
    'LanguageAdmired', 'LanguagesHaveEntry', 'LanguagesWantEntry',
    'DatabaseAdmired', 'DatabaseHaveEntry', 'DatabaseWantEntry',
    'PlatformAdmired', 'PlatformHaveEntry', 'PlatformWantEntry',
    'WebframeAdmired', 'WebframeHaveEntry', 'WebframeWantEntry',
    'DevEnvsAdmired', 'DevEnvHaveEntry', 'DevEnvWantEntry',
    'OpSysPersonal use', 'OpSysProfessional use',
    'OfficeStackAsyncAdmired', 'OfficeStackHaveEntry', 'OfficeStackWantEntry',
    'CommPlatformAdmired', 'CommPlatformHaveEntr', 'CommPlatformWantEntr',
    'AIModelsAdmired', 'AIModelsHaveEntry', 'AIModelsWantEntry',
    'AISent', 'AIAcc', 'AIComplex',
    'AIToolCurrently partially AI', "AIToolDon't plan to use AI for this task",
    'AIToolPlan to partially use AI', 'AIToolPlan to mostly use AI',
    'AIToolCurrently mostly AI', 'AIFrustration', 'AIExplain',
    'AIAgentChange', 'AgentUsesGeneral',
    'AIAgentImpactSomewhat agree', 'AIAgentImpactNeutral',
    'AIAgentImpactSomewhat disagree', 'AIAgentImpactStrongly agree',
    'AIAgentImpactStrongly disagree',
    'AIAgentChallengesNeutral', 'AIAgentChallengesSomewhat disagree',
    'AIAgentChallengesStrongly agree', 'AIAgentChallengesSomewhat agree',
    'AIAgentChallengesStrongly disagree',
    'AIAgentKnowledge', 'AIAgentKnowWrite',
    'AIAgentOrchestration', 'AIAgentOrchWrite',
    'AIAgentObserveSecure', 'AIAgentObsWrite',
    'AIAgentExternal', 'AIAgentExtWrite',
    'AIHuman', 'AIOpen', 'LanguageWantToWorkWith', 'DatabaseWantToWorkWith', 'PlatformWantToWorkWith', 'WebframeWantToWorkWith', 'DevEnvsWantToWorkWith', 'OfficeStackAsyncWantToWorkWith', 'AIModelsWantToWorkWith', 'CommPlatformWantToWorkWith'
]
prefixes_drop = ("TechEndorse", "TechOppose", "JobSatPoints", "SO")

multi_select_cols = [
    'LanguageHaveWorkedWith',
    'DatabaseHaveWorkedWith',
    'PlatformHaveWorkedWith',
    'WebframeHaveWorkedWith',
    'DevEnvsHaveWorkedWith',
    'OfficeStackAsyncHaveWorkedWith',
    'AIModelsHaveWorkedWith',
    'AIAgent_Uses'
]

In [6]:
def insert_mapped_column(df, base_col, new_col, mapping):
    if base_col not in df.columns:
        return
    df.insert(df.columns.get_loc(base_col) + 1, new_col, df[base_col].map(mapping))

def clean_text(s):
    if pd.isna(s):
        return ""
    s = str(s).strip()
    s = unicodedata.normalize("NFC", s)
    s = s.replace("–", "-").replace("—", "-").replace("’", "'")
    s = re.sub(r"\s+", " ", s)
    return s

def to_lowercase(s):
    if pd.isna(s):
        return s
    return str(s).lower()

def clean_multi_select_to_str(value):
    if pd.isna(value):
        return ""
    parts = [clean_text(p).lower() for p in str(value).split(";")]
    parts = sorted(set([p for p in parts if p]))
    return ";".join(parts)

def parse_years(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s == "":
        return np.nan
    try:
        return float(s)
    except:
        return np.nan


In [7]:
df = pd.read_csv(IN_PATH)

# nicht benötigte Spalten entfernen
df = df.drop(columns=drop_cols, errors="ignore")
df = df.drop(columns=df.columns[df.columns.str.startswith(prefixes_drop)], errors="ignore")

# YearsCode / WorkExp numersich
if "YearsCode" in df.columns:
    df["YearsCode"] = df["YearsCode"].apply(parse_years)
if "WorkExp" in df.columns:
    df["WorkExp"] = pd.to_numeric(df["WorkExp"], errors="coerce")

# lowercase
exclude_obj = {"Country"}
for col in df.select_dtypes(include=["object"]).columns:
    if col not in exclude_obj:
        df[col] = df[col].apply(to_lowercase)

# Multi-Select Cleanup
for col in [c for c in multi_select_cols if c in df.columns]:
    df[col] = df[col].apply(clean_multi_select_to_str)

# Plausabilitätsfilter
if {"WorkExp", "AgeNum"}.issubset(df.columns):
    df = df[~(df["WorkExp"] > (df["AgeNum"] - 16))].copy()
if {"YearsCode", "AgeNum"}.issubset(df.columns):
    df = df[~(df["YearsCode"] > (df["AgeNum"] - 6))].copy()

# 98%-Filter
for col in [c for c in ["ConvertedCompTotal", "WorkExp", "YearsCode"] if c in df.columns]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    q98 = df[col].quantile(0.98)
    before = len(df)
    df = df[df[col].isna() | (df[col] <= q98)].copy()
    print(f"{col}: kept <= q98={q98:.4g} | removed {before - len(df)} rows")

# Speichern
df.to_csv(OUT_PATH, index=False)
print("Saved:", df.shape, "->", OUT_PATH)


ConvertedCompTotal: kept <= q98=3.245e+05 | removed 314 rows
WorkExp: kept <= q98=35 | removed 386 rows
YearsCode: kept <= q98=40 | removed 336 rows
Saved: (19965, 38) -> survey_results_cleaned.csv


In [8]:
IN_PATH = "survey_results_cleaned.csv"
OUT_PATH = "survey_results_cleaned_final.csv"

df = pd.read_csv(IN_PATH)

thresh_rows = 32  # z.B. mindestens 32 ausgefüllte Spalten pro Zeile
df_rows = df.dropna(axis=0, thresh=thresh_rows)

print("vorher:", df.shape)
print("nachher:", df_rows.shape)
df_rows.info()
df_rows.to_csv(OUT_PATH, index=False)
print("Saved:", df_rows.shape, "->", OUT_PATH)

vorher: (19965, 38)
nachher: (18603, 38)
<class 'pandas.core.frame.DataFrame'>
Index: 18603 entries, 0 to 19964
Data columns (total 38 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Unnamed: 0                      18603 non-null  int64  
 1   ResponseId                      18603 non-null  int64  
 2   MainBranch                      18603 non-null  object 
 3   Age                             18603 non-null  object 
 4   EdLevel                         18592 non-null  object 
 5   Employment                      18603 non-null  object 
 6   Country                         18603 non-null  object 
 7   WorkExp                         18276 non-null  float64
 8   LearnCodeAI                     18589 non-null  object 
 9   YearsCode                       18554 non-null  float64
 10  DevType                         18603 non-null  object 
 11  OrgSize                         17326 non-null  object 
 

In [9]:
df_rows.ConvertedCompTotal.describe()

count     14290.000000
mean      90267.489323
std       61123.720567
min           0.000000
25%       46712.000000
50%       80000.000000
75%      120876.365727
max      323000.000000
Name: ConvertedCompTotal, dtype: float64